# Stage A: fixed-route adapter and rollout collection

Train a fresh fixed-route low-rank residual adapter and collect verified GSM8K rollouts. Export a checksummed handoff for Notebook 01b, where the route bank and router are trained.

**Inputs:** attach no previous checkpoints, norm files, or rollout files. This run needs internet access for the source repository and GSM8K dataset, plus a CUDA GPU.

## Stage A workflow

This notebook builds the fixed-route adapter from newly collected GSM8K training rollouts. Collection samples 5,000 unique questions with two initial completions each. Only questions with no strict-positive initial completion receive two targeted retries at a higher temperature and alternate prompt templates. The 0-shot prompt condition has the largest collection and training share. At most two distinct strict-positive completions per question become training rows.

The output `ars_fixed_adapter_bundle.zip` contains the fixed-route adapter, both calibration files, raw rollout shards, collection diagnostics, and pinned code snapshots. Notebook 01b consumes this handoff to train the steering bank and router. No previous model checkpoints or rollout files are loaded.

In [ ]:
import codecs
import json
import os
import selectors
import subprocess
import sys
import time
from pathlib import Path

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONUNBUFFERED"] = "1"

RUNTIME_PLATFORM = "auto"  # Choose auto, kaggle, or colab.
REPO_URL = "https://github.com/Abdullah182155/adaptive-residual-steering.git"
if RUNTIME_PLATFORM == "auto":
    if Path("/kaggle/working").is_dir():
        RUNTIME_PLATFORM = "kaggle"
    elif Path("/content").is_dir():
        RUNTIME_PLATFORM = "colab"
    else:
        raise ValueError("Could not auto-detect Kaggle or Colab; set RUNTIME_PLATFORM explicitly")
if RUNTIME_PLATFORM == "kaggle":
    WORK_ROOT = Path("/kaggle/working")
elif RUNTIME_PLATFORM == "colab":
    WORK_ROOT = Path("/content")
else:
    raise ValueError("Could not detect Kaggle or Colab; set RUNTIME_PLATFORM explicitly")
REPO_DIR = WORK_ROOT / "adaptive-residual-steering"
BRANCH = "development"

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
required_handoff_sources = (
    "scripts/run_session3_router.py",
    "scripts/run_session3_mcq_benchmark.py",
    "scripts/research_notebook_utils.py",
    "scripts/research_notebook_analysis.py",
    "scripts/run_ars_pipeline.py",
    "scripts/run_rft_pipeline.py",
    "src/configs/config.py",
    "src/data/dataset.py",
    "src/evaluation/answers.py",
    "src/evaluation/benchmark.py",
    "src/evaluation/generation.py",
    "src/evaluation/ood.py",
    "src/models/wrapper.py",
    "src/routing/adaptive_subset_router.py",
    "src/steering/route_bank.py",
    "src/steering/steernet.py",
    "src/training/adaptive_data.py",
    "src/training/adaptive_router_training.py",
    "src/training/adaptive_subset_training.py",
    "src/training/batching.py",
    "src/training/checkpoints.py",
    "src/training/optimization.py",
    "src/training/rft_collect.py",
    "src/training/rft_trainer.py",
    "src/training/route_curriculum.py",
    "src/utils/runtime.py",
    "src/steering/__init__.py",
    "src/training/__init__.py",
    "src/models/__init__.py",
    "src/evaluation/__init__.py",
    "src/evaluation/chain.py",
    "src/models/loading.py",
)
missing_handoff_sources = [relative_path for relative_path in required_handoff_sources if not (REPO_DIR / relative_path).is_file()]
if missing_handoff_sources:
    raise RuntimeError(
        f"The {BRANCH} source is missing files required to finish and package Stage A: {missing_handoff_sources}. "
        "Add and push these source files, then rerun this setup cell before starting GPU work."
    )
required_cli_flags = {
    "scripts/run_rft_pipeline.py": (
        "--retry_k",
        "--retry_temperature",
        "--collection_shot_weights",
        "--diversify_collection_templates",
        "--max_rft_rows_per_question",
        "--steering_architecture",
        "fixed_route_low_rank_residual",
        "--quality_probe_nonblocking",
        "rollout_quality_probe",
    ),
    "scripts/run_session3_router.py": (
        "--max_rows_per_question",
        "--fixed_adapter_checkpoint",
        "--fixed_adapter_norm_calibration",
        "--route_examples_per_expert",
        "--screen_validation_rows",
        "--max_pair_routes",
        "--max_triple_routes",
        "--max_quad_routes",
        "--min_layer_gap",
        "--control_gap",
        "--spacing_control_limit",
    ),
}
for script, required_flags in required_cli_flags.items():
    help_result = subprocess.run(
        [sys.executable, "-u", script, "--help"],
        cwd=REPO_DIR,
        capture_output=True,
        text=True,
        check=True,
    )
    missing_flags = [flag for flag in required_flags if flag not in help_result.stdout]
    if missing_flags:
        raise RuntimeError(
            f"The {BRANCH} source is missing notebook-required CLI flags in {script}: {missing_flags}. "
            "Push the matching repository changes and rerun this setup cell."
        )
print(f"Runtime: {RUNTIME_PLATFORM}; repository: {REPO_DIR} ({BRANCH}); CLI compatibility: passed", flush=True)

def _emit_stream_line(state, line, *, force=False):
    """Print child output snapshots as normal notebook lines with a short throttle."""
    now = time.monotonic()
    if line and (force or now - state["last_update"] >= 1.0 or "100%" in line):
        print(f"{state['prefix']}{line}", flush=True)
        state["last_update"] = now


def _consume_stream_chunk(state, chunk):
    """Split child output on newline and tqdm carriage-return boundaries."""
    state["pending"] += state["decoder"].decode(chunk)
    while "\r" in state["pending"] or "\n" in state["pending"]:
        cr = state["pending"].find("\r") if "\r" in state["pending"] else len(state["pending"])
        lf = state["pending"].find("\n") if "\n" in state["pending"] else len(state["pending"])
        boundary = min(cr, lf)
        separator = state["pending"][boundary]
        line, state["pending"] = state["pending"][:boundary], state["pending"][boundary + 1:]
        _emit_stream_line(state, line, force=(separator == "\n" or "100%" in line))


def _finish_stream(state):
    state["pending"] += state["decoder"].decode(b"", final=True)
    _emit_stream_line(state, state["pending"], force=True)


def _start_live_process(command, env=None):
    return subprocess.Popen(
        command,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=0,
    )


def run_live(command, *, env=None, prefix=""):
    """Run one child and render its tqdm refreshes as flushed notebook output."""
    run_parallel_live([(prefix, command, env)])


def run_parallel_live(jobs):
    """Run child commands concurrently and stream each labelled pipe in this thread."""
    selector = selectors.DefaultSelector()
    processes = []
    for prefix, command, env in jobs:
        process = _start_live_process(command, env)
        processes.append((prefix, command, process))
        state = {
            "prefix": prefix,
            "decoder": codecs.getincrementaldecoder("utf-8")(errors="replace"),
            "pending": "",
            "last_update": 0.0,
        }
        selector.register(process.stdout, selectors.EVENT_READ, state)

    while selector.get_map():
        for key, _ in selector.select(timeout=0.25):
            chunk = os.read(key.fileobj.fileno(), 4096)
            if chunk:
                _consume_stream_chunk(key.data, chunk)
            else:
                _finish_stream(key.data)
                selector.unregister(key.fileobj)
                key.fileobj.close()

    selector.close()
    failures = []
    for prefix, command, process in processes:
        return_code = process.wait()
        if return_code:
            failures.append(f"{prefix.strip()} exited with code {return_code}: {command}")
    if failures:
        raise RuntimeError("\n".join(failures))

## 2. Fresh-run configuration

This stage trains the fixed-route low-rank residual adapter and steering bank from newly collected data. It saves a handoff artifact before the separate router stage so a long router run never forces recollection or retraining.

In [ ]:
import json
from datetime import datetime
from pathlib import Path
import os
import sys

SMOKE_TEST = False
GPU_MODE = "auto"  # Choose auto, single (Colab), or dual (Kaggle Dual T4).
ATTENTION_IMPLEMENTATION = "sdpa"
os.environ["ARS_ATTENTION_IMPLEMENTATION"] = ATTENTION_IMPLEMENTATION
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONUNBUFFERED"] = "1"

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = Path("report/fixed_steering_fresh") / RUN_ID
RFT_DIR = RUN_DIR / "rft"
NORM_FILE = RUN_DIR / "fixed_route_p99_norms.json"
CHECKPOINT = RUN_DIR / "fixed_route_adapter.pt"

FIXED_ROUTE = [12, 15, 18, 21]
SEED = 42
if SMOKE_TEST:
    CALIB_N, EQUIVALENCE_N, QUALITY_PROBE_N = 24, 5, 5
    COLLECTION_QUESTIONS, COLLECTION_K, COLLECTION_RETRY_K = 40, 2, 2
    TRAIN_EPOCHS = 1
else:
    CALIB_N, EQUIVALENCE_N, QUALITY_PROBE_N = 128, 100, 60
    COLLECTION_QUESTIONS, COLLECTION_K, COLLECTION_RETRY_K = 5000, 2, 2
    TRAIN_EPOCHS = 2
COLLECTION_SHOT_WEIGHTS = [0.60, 0.25, 0.15]  # 0/2/4-shot rollout prompts
COLLECTION_RETRY_TEMPERATURE = 0.9
COLLECTION_TARGET_POSITIVES = 2
MAX_RFT_ROWS_PER_QUESTION = 2
TRAIN_SHOT_WEIGHTS = [0.60, 0.25, 0.15]  # 0/2/4-shot training prompts
PHASE_STATUS = {name: "pending" for name in ("runtime", "tests", "norms", "equivalence", "rollout_quality_probe", "collection", "training", "alpha_calibration", "candidate_norms", "steering_bank", "router", "packaging")}
RUN_DIR.mkdir(parents=True, exist_ok=False)
RFT_DIR.mkdir(parents=True, exist_ok=True)
COLLECTION_POLICY = {
    "questions": COLLECTION_QUESTIONS,
    "initial_rollouts_per_question": COLLECTION_K,
    "retry_rollouts_per_question": COLLECTION_RETRY_K,
    "retry_condition": "zero strict-positive samples among initial attempts",
    "retry_temperature": COLLECTION_RETRY_TEMPERATURE,
    "target_distinct_positive_rows_per_question": COLLECTION_TARGET_POSITIVES,
    "max_training_rows_per_question": MAX_RFT_ROWS_PER_QUESTION,
    "shot_weights_0_2_4": COLLECTION_SHOT_WEIGHTS,
    "diversified_prompt_templates": True,
    "initial_temperature": 0.7,
    "top_p": 0.95,
    "max_new_tokens": 200,
    "max_prompt_tokens": 768,
    "positive_deduplication": "reasoning_key",
}
COLLECTION_PLAN = RFT_DIR / "collection_plan.json"
COLLECTION_PLAN.write_text(json.dumps(COLLECTION_POLICY, indent=2) + "\n", encoding="utf-8")
print({"run_dir": str(RUN_DIR), "fresh_artifacts_only": True, "route": FIXED_ROUTE, "calibration_questions": CALIB_N, "collection_policy": COLLECTION_POLICY, "epochs": TRAIN_EPOCHS, "micro_batch": 1, "grad_accum": 32, "candidate_norm_calibration": True, "alpha_calibration": "deferred_to_notebook_01b"}, flush=True)


## 3. Runtime and tests

Confirm GPU visibility and run the repository test suite. Failures are recorded so the notebook can report the complete run state.


In [ ]:
try:
    gpu_query = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True, check=True)
    visible_gpus = [line.strip() for line in gpu_query.stdout.splitlines() if line.strip()]
    print("Visible GPUs:", visible_gpus, flush=True)
    if GPU_MODE not in {"auto", "single", "dual"}:
        raise ValueError("GPU_MODE must be auto, single, or dual")
    if not visible_gpus:
        GPU_COUNT = 0
    elif GPU_MODE == "single":
        GPU_COUNT = 1
    elif GPU_MODE == "dual" and len(visible_gpus) >= 2:
        GPU_COUNT = 2
    else:
        GPU_COUNT = min(len(visible_gpus), 2)
    print({"gpu_mode": GPU_MODE, "gpus_used_for_collection": GPU_COUNT}, flush=True)
    PHASE_STATUS["runtime"] = "passed" if GPU_COUNT >= 2 else "warning_single_gpu" if GPU_COUNT == 1 else "failed_no_gpu"
except Exception as exc:
    visible_gpus = []
    PHASE_STATUS["runtime"] = f"diagnostic_failed: {type(exc).__name__}: {exc}"

try:
    test_result = subprocess.run([sys.executable, "-u", "-m", "pytest", "tests/", "-x", "-q"], check=False)
    PHASE_STATUS["tests"] = "passed" if test_result.returncode == 0 else f"failed_exit_{test_result.returncode}"
except Exception as exc:
    PHASE_STATUS["tests"] = f"failed: {type(exc).__name__}: {exc}"
print("Test status:", PHASE_STATUS["tests"], flush=True)

## 4. Calibrate fresh fixed-route layer norms

Calibration reads GSM8K training questions only and writes a new run-specific p99 file. No uploaded calibration is consulted.


In [ ]:
try:
    run_live([sys.executable, "-u", "scripts/calibrate_norms.py", "--n_samples", str(CALIB_N), "--seed", str(SEED), "--candidate_layers", *map(str, FIXED_ROUTE), "--max_length", "768", "--output", str(NORM_FILE)], prefix="[norms] ")
    PHASE_STATUS["norms"] = "passed" if NORM_FILE.is_file() else "failed_missing_output"
except Exception as exc:
    PHASE_STATUS["norms"] = f"failed: {type(exc).__name__}: {exc}"
    print("Fresh norm calibration failed; later dependent stages will record skips.", flush=True)

## 5. Equivalence and Rollout quality probe diagnostics

Batched generation equivalence is diagnostic for this run. Rollout quality probe records sampled-vs-greedy quality but is nonblocking so a small or noisy probe does not terminate the notebook.


In [ ]:
try:
    run_live([sys.executable, "-u", "scripts/run_rft_pipeline.py", "--stage", "equivalence", "--steering_architecture", "fixed_route_low_rank_residual", "--norm_calibration", str(NORM_FILE), "--n_questions", str(EQUIVALENCE_N), "--output_dir", str(RFT_DIR)], prefix="[equivalence] ")
    PHASE_STATUS["equivalence"] = "completed_diagnostic"
except Exception as exc:
    PHASE_STATUS["equivalence"] = f"diagnostic_failed: {type(exc).__name__}: {exc}"
    print("Equivalence diagnostic did not pass; continuing with batch-1 training/evaluation paths.", flush=True)

if NORM_FILE.is_file():
    try:
        run_live([sys.executable, "-u", "scripts/run_rft_pipeline.py", "--stage", "rollout_quality_probe", "--steering_architecture", "fixed_route_low_rank_residual", "--norm_calibration", str(NORM_FILE), "--n_questions", str(QUALITY_PROBE_N), "--k", "8", "--temperature", "0.7", "--quality_probe_nonblocking", "--output_dir", str(RFT_DIR)], prefix="[rollout_quality_probe] ")
        PHASE_STATUS["rollout_quality_probe"] = "completed_diagnostic"
    except Exception as exc:
        PHASE_STATUS["rollout_quality_probe"] = f"failed: {type(exc).__name__}: {exc}"
        print("Rollout quality probe failed; rollout workers will still be attempted and their outcome recorded.", flush=True)
else:
    PHASE_STATUS["rollout_quality_probe"] = "skipped_missing_fresh_norms"

## 6. Collect new positive rollouts on both T4s

Both workers sample disjoint portions of one deterministic question set. Each shard writes new JSONL and stats into this run directory. Low yield is recorded rather than raised; training separately reports if no usable positives exist.


In [ ]:
if NORM_FILE.is_file():
    collection_env = os.environ.copy()
    jobs = []
    for shard in (0, 1):
        gpu = str(shard) if GPU_COUNT >= 2 else "0"
        env = collection_env.copy()
        env["CUDA_VISIBLE_DEVICES"] = gpu
        command = [sys.executable, "-u", "scripts/run_rft_pipeline.py", "--stage", "collect", "--steering_architecture", "fixed_route_low_rank_residual", "--norm_calibration", str(NORM_FILE), "--n_questions", str(COLLECTION_QUESTIONS), "--k", str(COLLECTION_K), "--temperature", "0.7", "--retry_k", str(COLLECTION_RETRY_K), "--retry_temperature", str(COLLECTION_RETRY_TEMPERATURE), "--target_positive_samples", str(COLLECTION_TARGET_POSITIVES), "--collection_shot_weights", *map(str, COLLECTION_SHOT_WEIGHTS), "--diversify_collection_templates", "--n_shards", "2", "--shard", str(shard), "--allow_low_yield", "--output_dir", str(RFT_DIR)]
        jobs.append((f"[GPU{gpu}/shard{shard}] ", command, env))
    try:
        if GPU_COUNT >= 2:
            print("Collecting rollout shards in parallel on two GPUs.", flush=True)
            run_parallel_live(jobs)
        else:
            print("One GPU detected; collecting both rollout shards sequentially on GPU 0.", flush=True)
            for job in jobs:
                run_live(job[1], env=job[2], prefix=job[0])
        PHASE_STATUS["collection"] = "workers_completed"
        shards = [RFT_DIR / f"shard{i}.jsonl" for i in (0, 1)]
    except Exception as exc:
        existing = [p.name for p in RFT_DIR.glob("shard*.jsonl") if p.is_file() and p.stat().st_size]
        PHASE_STATUS["collection"] = f"partial_or_failed: {type(exc).__name__}: {exc}; available={existing}"
else:
    PHASE_STATUS["collection"] = "skipped_missing_fresh_norms"

# Validate both shards and produce a compact, auditable collection report.
import collections
from src.training.rft_collect import reasoning_key

collection_worker_status = PHASE_STATUS["collection"]
collection_rows = []
collection_errors = []
shard_summaries = []
seen_question_indices = set()
for shard in (0, 1):
    shard_path = RFT_DIR / f"shard{shard}.jsonl"
    stats_path = RFT_DIR / f"shard{shard}_stats.json"
    expected_scheduled = len(range(shard, COLLECTION_QUESTIONS, 2))
    if not shard_path.is_file() or not stats_path.is_file():
        collection_errors.append(f"shard{shard}: missing rollout or stats file")
        continue
    try:
        stats = json.loads(stats_path.read_text(encoding="utf-8"))
    except Exception as exc:
        collection_errors.append(f"shard{shard}: invalid stats JSON: {type(exc).__name__}: {exc}")
        continue
    shard_rows = []
    try:
        with shard_path.open(encoding="utf-8") as stream:
            for line_number, line in enumerate(stream, start=1):
                if not line.strip():
                    continue
                record = json.loads(line)
                question_index = int(record["question_index"])
                # question_index is the source dataset row ID, not the sampled position.
                if question_index < 0:
                    collection_errors.append(f"shard{shard}:{line_number}: question index must be nonnegative")
                if question_index in seen_question_indices:
                    collection_errors.append(f"duplicate question_index={question_index}")
                seen_question_indices.add(question_index)
                samples = record.get("samples", [])
                initial = [sample for sample in samples if sample.get("phase") == "initial"]
                retries = [sample for sample in samples if sample.get("phase") == "targeted_retry"]
                if int(record.get("initial_k", -1)) != COLLECTION_K or len(initial) != COLLECTION_K:
                    collection_errors.append(f"shard{shard}:{line_number}: expected {COLLECTION_K} initial attempts")
                if sorted(int(sample.get("attempt_index", -1)) for sample in initial) != list(range(COLLECTION_K)):
                    collection_errors.append(f"shard{shard}:{line_number}: initial attempt indexes are missing or duplicated")
                retry_skips = int(record.get("retry_prompt_skips", 0))
                retry_requested = int(record.get("retry_k_requested", 0))
                if retry_requested and len(retries) + retry_skips != retry_requested:
                    collection_errors.append(f"shard{shard}:{line_number}: retry attempts and prompt skips do not match the requested count")
                if retries and any(sample.get("strict_ok") for sample in initial):
                    collection_errors.append(f"shard{shard}:{line_number}: retry ran despite an initial strict positive")
                if record.get("retry_trigger") and any(sample.get("strict_ok") for sample in initial):
                    collection_errors.append(f"shard{shard}:{line_number}: retry trigger conflicts with initial outcomes")
                if not record.get("retry_trigger") and retries:
                    collection_errors.append(f"shard{shard}:{line_number}: retries exist without a retry trigger")
                shard_rows.append(record)
    except Exception as exc:
        collection_errors.append(f"shard{shard}: invalid JSONL: {type(exc).__name__}: {exc}")
    if len(shard_rows) != int(stats.get("n", -1)):
        collection_errors.append(f"shard{shard}: JSONL count does not match saved stats")
    if int(stats.get("n", 0)) + int(stats.get("skipped_long_prompts", 0)) != expected_scheduled:
        collection_errors.append(f"shard{shard}: completed plus skipped questions does not match its assigned count")
    collection_rows.extend(shard_rows)
    shard_summaries.append({"shard": shard, "expected_questions": expected_scheduled, "stats": stats})

shot_question_counts = collections.Counter()
attempt_shot_counts = collections.Counter()
template_attempt_counts = collections.Counter()
strict_positive_samples = 0
initial_positive_questions = 0
final_positive_questions = 0
questions_with_two_unique_positives = 0
retry_question_count = 0
retry_sample_count = 0
retry_prompt_skip_count = 0
for record in collection_rows:
    shot_question_counts[str(record.get("n_shot", "unknown"))] += 1
    samples = record.get("samples", [])
    initial = [sample for sample in samples if sample.get("phase") == "initial"]
    retries = [sample for sample in samples if sample.get("phase") == "targeted_retry"]
    retry_question_count += int(bool(retries))
    retry_sample_count += len(retries)
    retry_prompt_skip_count += int(record.get("retry_prompt_skips", 0))
    initial_positive_questions += int(any(sample.get("strict_ok") for sample in initial))
    positives = [sample for sample in samples if sample.get("strict_ok")]
    final_positive_questions += int(bool(positives))
    strict_positive_samples += len(positives)
    unique_positive_signatures = {reasoning_key(sample.get("text", "")) for sample in positives}
    questions_with_two_unique_positives += int(len(unique_positive_signatures) >= COLLECTION_TARGET_POSITIVES)
    for sample in samples:
        attempt_shot_counts[str(sample.get("n_shot", "unknown"))] += 1
        template_attempt_counts[str(sample.get("template_id", "unknown"))] += 1

collection_summary = {
    "policy": COLLECTION_POLICY,
    "worker_status_before_validation": collection_worker_status,
    "questions_written": len(collection_rows),
    "unique_question_indices": len(seen_question_indices),
    "duplicate_question_indices": len(collection_rows) - len(seen_question_indices),
    "initial_positive_questions": initial_positive_questions,
    "final_positive_questions": final_positive_questions,
    "strict_positive_samples": strict_positive_samples,
    "questions_with_two_unique_positives": questions_with_two_unique_positives,
    "retry_questions": retry_question_count,
    "retry_samples": retry_sample_count,
    "retry_prompt_skips": retry_prompt_skip_count,
    "questions_unresolved_after_retry": len(collection_rows) - final_positive_questions,
    "question_shot_counts": dict(shot_question_counts),
    "attempt_shot_counts": dict(attempt_shot_counts),
    "template_attempt_counts": dict(template_attempt_counts),
    "shards": shard_summaries,
    "validation_errors": collection_errors,
}
COLLECTION_SUMMARY = RFT_DIR / "collection_summary.json"
COLLECTION_SUMMARY.write_text(json.dumps(collection_summary, indent=2) + "\n", encoding="utf-8")
collection_ready = (
    collection_worker_status == "workers_completed"
    and not collection_errors
    and len(collection_rows) == COLLECTION_QUESTIONS
    and len(collection_rows) == len(seen_question_indices)
    and sum(int(item["stats"].get("n", 0)) for item in shard_summaries) == COLLECTION_QUESTIONS
    and sum(int(item["stats"].get("skipped_long_prompts", 0)) for item in shard_summaries) == 0
)
PHASE_STATUS["collection"] = "completed" if collection_ready else "incomplete_or_invalid_outputs"
print(json.dumps({"collection_ready": collection_ready, **collection_summary}, indent=2), flush=True)


Training uses only the newly collected and validated rollout shards, route `[12, 15, 18, 21]`, rank 16, p99 norm caps, identity magnitude, and positive-completion token-mean NLL. For each question, two initial completions are sampled; only questions with zero strict-positive initial completions receive two retries using alternate prompt templates at temperature 0.9. The training-row builder caps data at two distinct strict-positive completions per question and uses shot weights `[0.60, 0.25, 0.15]` for 0/2/4-shot prompts. The frozen Phi-2 backbone is not updated.


In [ ]:
shards_ready = collection_ready and all((RFT_DIR / f"shard{i}.jsonl").is_file() for i in (0, 1))
if NORM_FILE.is_file() and shards_ready:
    train_env = os.environ.copy()
    train_env["CUDA_VISIBLE_DEVICES"] = "0"
    try:
        run_live([sys.executable, "-u", "scripts/run_rft_pipeline.py", "--stage", "train", "--steering_architecture", "fixed_route_low_rank_residual", "--norm_calibration", str(NORM_FILE), "--epochs", str(TRAIN_EPOCHS), "--batch_size", "1", "--grad_accum", "32", "--checkpoint", str(CHECKPOINT), "--output_dir", str(RFT_DIR), "--train_shot_weights", *map(str, TRAIN_SHOT_WEIGHTS), "--max_rft_rows_per_question", str(MAX_RFT_ROWS_PER_QUESTION), "--diversify_prompt_templates"], env=train_env, prefix="[train] ")
        PHASE_STATUS["training"] = "passed" if CHECKPOINT.is_file() else "failed_missing_checkpoint"
    except Exception as exc:
        PHASE_STATUS["training"] = f"failed: {type(exc).__name__}: {exc}"
        print("Training failed; intensity calibration and test evaluation will be skipped.", flush=True)
else:
    PHASE_STATUS["training"] = "skipped_missing_norms_or_complete_valid_collection"

## 8. Defer steering-intensity calibration to Notebook 01b

Intensity calibration uses the saved fixed adapter and validation questions. It runs in Notebook 01b alongside bank and router training to balance the GPU workload.

In [ ]:
PHASE_STATUS["alpha_calibration"] = "deferred_to_notebook_01b"
print("Steering-intensity calibration is deferred to Notebook 01b.", flush=True)

## 9. Candidate-layer norm calibration

Uses prompts from the newly collected rollouts and produces p99 references for all bank layers.


In [ ]:
CANDIDATE_LAYERS = [8,11,12,13,15,17,18,19,20,21,22,23,24,26]
ROLLOUT_FILES = [RFT_DIR / "shard0.jsonl", RFT_DIR / "shard1.jsonl"]
CANDIDATE_NORMS = RUN_DIR / "candidate_norms_p99.json"
if collection_ready and all(p.is_file() and p.stat().st_size for p in ROLLOUT_FILES):
    try:
        run_live([sys.executable,"-u","scripts/calibrate_norms.py","--n_samples",str(CALIB_N),"--seed",str(SEED),"--candidate_layers",*map(str,CANDIDATE_LAYERS),"--max_length","768","--rollout_files",*map(str,ROLLOUT_FILES),"--output",str(CANDIDATE_NORMS)],prefix="[candidate-norms] ")
        PHASE_STATUS["candidate_norms"] = "passed" if CANDIDATE_NORMS.is_file() else "failed_missing_output"
    except Exception as exc:
        PHASE_STATUS["candidate_norms"] = f"failed: {type(exc).__name__}: {exc}"
else:
    PHASE_STATUS["candidate_norms"] = "skipped_missing_or_invalid_rollouts"
print(PHASE_STATUS["candidate_norms"], flush=True)

## 10. Defer route-bank training to Notebook 01b

Candidate-layer norms are calibrated here and saved with the rollouts. The independent route experts are trained in Notebook 01b alongside the router, balancing the GPU workload across the two sessions.

In [ ]:
PHASE_STATUS["steering_bank"] = "deferred_to_notebook_01b"
print("Steering-bank training is deferred to Notebook 01b.", flush=True)

## 11. Save a fixed-adapter handoff

The fixed-route adapter, its calibration, candidate-layer norms, and validated fresh rollouts are ready for Notebook 01b. That notebook trains the route bank first, then trains the router from the resulting bank.

In [ ]:
PHASE_STATUS["router"] = "deferred_to_notebook_01b"
print("Route-bank and router training are deferred to Notebook 01b.", flush=True)

## 12. Package the fixed-adapter handoff

The handoff is ready when the fixed-route adapter, both norm files, both complete rollout shards, required code snapshots, and Stage A prerequisite statuses are present.

In [ ]:
import hashlib
import zipfile
from datetime import datetime, timezone


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

required = {
    "checkpoints/fixed_route_adapter.pt": CHECKPOINT,
    "calibration/fixed_route_p99_norms.json": NORM_FILE,
    "calibration/candidate_norms_p99.json": CANDIDATE_NORMS,
}
available = {name: path for name, path in required.items() if path.is_file()}
required_code_paths = (
    "scripts/run_session3_router.py",
    "scripts/run_session3_mcq_benchmark.py",
    "scripts/research_notebook_utils.py",
    "scripts/research_notebook_analysis.py",
    "scripts/run_ars_pipeline.py",
    "scripts/run_rft_pipeline.py",
    "scripts/calibrate_norms.py",
    "src/configs/config.py",
    "src/data/dataset.py",
    "src/evaluation/answers.py",
    "src/evaluation/benchmark.py",
    "src/evaluation/generation.py",
    "src/evaluation/ood.py",
    "src/models/wrapper.py",
    "src/routing/adaptive_subset_router.py",
    "src/steering/route_bank.py",
    "src/steering/steernet.py",
    "src/training/adaptive_data.py",
    "src/training/adaptive_router_training.py",
    "src/training/adaptive_subset_training.py",
    "src/training/batching.py",
    "src/training/checkpoints.py",
    "src/training/optimization.py",
    "src/training/rft_collect.py",
    "src/training/rft_trainer.py",
    "src/training/route_curriculum.py",
    "src/utils/runtime.py",
    "src/steering/__init__.py",
    "src/training/__init__.py",
    "src/models/__init__.py",
    "src/evaluation/__init__.py",
    "src/evaluation/chain.py",
    "src/models/loading.py",
)
code_files = {
    f"code/{relative_path}": REPO_DIR / relative_path
    for relative_path in required_code_paths
    if (REPO_DIR / relative_path).is_file()
}
rollout_paths = [RFT_DIR / f"shard{i}.jsonl" for i in (0, 1)]
rollouts = {
    f"training/rollouts/{path.name}": {"sha256": sha256_file(path)}
    for path in rollout_paths
    if path.is_file() and path.stat().st_size
}
missing = [name for name, path in required.items() if not path.is_file()]
missing_rollouts = [path.name for path in rollout_paths if not path.is_file() or not path.stat().st_size]
missing_code = [path for path in required_code_paths if f"code/{path}" not in code_files]
required_phases = {
    "runtime": {"passed", "warning_single_gpu"},
    "tests": {"passed"},
    "norms": {"passed"},
    "collection": {"completed"},
    "training": {"passed"},
    "candidate_norms": {"passed"},
}
failed_phases = {
    name: PHASE_STATUS.get(name, "not_run")
    for name, accepted in required_phases.items()
    if PHASE_STATUS.get(name) not in accepted
}
fixed_adapter_ready = not missing and not missing_rollouts and not missing_code and not failed_phases
PHASE_STATUS["packaging"] = "passed" if fixed_adapter_ready else f"partial: missing={missing + missing_rollouts + missing_code}, phases={failed_phases}"
commit = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, capture_output=True, text=True, check=True).stdout.strip()
manifest = {
    "format": "ars_fixed_adapter_handoff_v1",
    "architecture_id": "ars.fixed_route_low_rank_residual",
    "architecture_version": "1.0.0",
    "fixed_adapter_ready": fixed_adapter_ready,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "run_id": RUN_ID,
    "code_commit": commit,
    "model": "microsoft/phi-2",
    "route": FIXED_ROUTE,
    "candidate_layers": CANDIDATE_LAYERS,
    "adapter_class": "fixed_route_low_rank_residual",
    "rank": 16,
    "rho_max": 0.10,
    "norm_stat": "p99_norm",
    "seed": SEED,
    "shot_weights": TRAIN_SHOT_WEIGHTS,
    "collection_policy": COLLECTION_POLICY,
    "phase_status": PHASE_STATUS,
    "required_phase_status": {name: PHASE_STATUS.get(name) for name in required_phases},
    "missing_required": missing + missing_rollouts + missing_code,
    "artifacts": {name: {"sha256": sha256_file(path)} for name, path in available.items()},
    "collection_metadata_hashes": {
        "training/collection_plan.json": sha256_file(COLLECTION_PLAN),
        "training/collection_summary.json": sha256_file(COLLECTION_SUMMARY),
    },
    "rollouts": rollouts,
    "code": {name: {"sha256": sha256_file(path)} for name, path in code_files.items()},
}
manifest_path = RUN_DIR / "fixed_adapter_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
status_path = RUN_DIR / "phase_status.json"
status_path.write_text(json.dumps(PHASE_STATUS, indent=2) + "\n", encoding="utf-8")
bundle = RUN_DIR / "ars_fixed_adapter_bundle.zip"
with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write(manifest_path, "manifest.json")
    archive.write(status_path, "training/phase_status.json")
    for path, archive_name in ((COLLECTION_PLAN, "training/collection_plan.json"), (COLLECTION_SUMMARY, "training/collection_summary.json")):
        if path.is_file():
            archive.write(path, archive_name)
    for name, path in available.items():
        archive.write(path, name)
    for name, path in code_files.items():
        archive.write(path, name)
    for name in rollouts:
        archive.write(RFT_DIR / Path(name).name, name)
    for path in sorted(RFT_DIR.glob("*stats.json")):
        archive.write(path, f"training/rollouts/{path.name}")
    for name in ("rft_training_log.json",):
        path = RFT_DIR / name
        if path.is_file():
            archive.write(path, f"training/{name}")
print(json.dumps({
    "bundle": str(bundle),
    "bundle_exists": bundle.is_file(),
    "fixed_adapter_ready": fixed_adapter_ready,
    "phase_status": PHASE_STATUS,
    "failed_phases": failed_phases,
    "included_artifacts": list(available),
    "included_rollouts": list(rollouts),
    "included_code": list(code_files),
    "missing": missing + missing_rollouts + missing_code,
}, indent=2), flush=True)